# Prerequisites

## Transaction 01 - Create Delta Table

In [0]:
%sql
USE CATALOG wns24082026;
DROP TABLE IF EXISTS quickstart_schema.users;
CREATE TABLE IF NOT EXISTS quickstart_schema.users (
    id INTEGER,
    name STRING,
    dob DATE,
    email STRING,
    gender STRING,
    country STRING,
    region STRING,
    city STRING,
    asset INTEGER,
    marital_status STRING
  );

DESCRIBE EXTENDED quickstart_schema.users

# Transaction 02 - Load 'users_001.csv' into Delta Table

In [0]:
df = spark.read.csv(
    path="/Volumes/wns24082026/quickstart_schema/sandbox/datasets/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
)
df.write.saveAsTable("quickstart_schema.users", mode="OVERWRITE")

# [Transaction 03] - Filter country = 'India'

In [0]:
from pyspark.sql.functions import col
df.filter(col("country")=="India").write.saveAsTable("quickstart_schema.users", mode="OVERWRITE")

# [Transaction 04] = Filter country = 'United States'

In [0]:
df.filter(col("country")=="United States").write.saveAsTable("quickstart_schema.users", mode="OVERWRITE")

# List Transactions

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark, "quickstart_schema.users"
)
delta_table.history().display()

# [Versioning]

## PySpark

In [0]:
spark.read.option("versionAsOf", 2).table("quickstart_schema.users").display()

## Using SQL

In [0]:
%sql

SELECT * from quickstart_schema.users VERSION AS OF 2;

# TIMESTAMP

In [0]:
spark.read.option("timestampAsOf", "2026-08-28T06:09:38").table("quickstart_schema.users").display()

In [0]:
%sql

SELECT * from quickstart_schema.users TIMESTAMP AS OF '2026-08-28T06:09:38' LIMIT 4;

# [Advantage] - Restore

## Delta Table API

In [0]:
from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark,"quickstart_schema.users")
delta_table.restoreToVersion(2)

## SQL

In [0]:
%sql

RESTORE TABLE quickstart_schema.users TO VERSION AS OF 1;